# Análise complementar do modelo de risco

Análises que **não** fazem parte da execução padrão do `cvt --risk` (que roda os
modelos clássicos + critérios separados). Aqui ficam duas análises complementares,
que antes eram flags (`--importance`, `--optimize`) e foram movidas para cá:

1. **Importância de features** (XGBoost gain) + ablação das features de Monte Carlo.
2. **Otimização de hiperparâmetros** com Optuna (XGBoost, RandomForest, LogReg).

Ambas usam o alvo de risco (`manobra_combinado_curva`) e a whitelist de features da
flag `risk` (definida em `features.yaml`). Os gráficos são salvos em `results/risco/`.

In [ ]:
# Garante que o diretório de trabalho é a raiz do projeto (onde estão config.yaml e data/).
import os
from pathlib import Path

_p = Path.cwd()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('raiz do projeto:', _p)

In [ ]:
from curvant.utils.config import carregar_config, carregar_features_config
from curvant.driving.features import configurar_features_ativas, resolver_features_flag
from curvant.cli import _carregar_features
from curvant.steps import train_risk

cfg = carregar_config()
feat_cfg = carregar_features_config()

# Carrega o features_df do cache (ou reconstrói o pipeline se não houver cache).
_, features_df = _carregar_features(cfg, feat_cfg, rebuild=False, mostrar_risco=True)

# Instala a whitelist da flag 'risk' — as análises abaixo usam colunas_features().
configurar_features_ativas(resolver_features_flag(feat_cfg, 'risk'))
print('features ativas (risk):', len(resolver_features_flag(feat_cfg, 'risk')))

## 1. Importância de features

Treina um XGBoost sobre o alvo de risco, imprime a importância (gain) por feature e
faz a ablação automática das features de Monte Carlo (`mc_p_*`), comparando as métricas
com e sem elas. O gráfico é salvo em `results/risco/importancia_features.pdf`.

In [ ]:
train_risk.importancia_features(features_df, cfg)

## 2. Otimização de hiperparâmetros (Optuna)

Tuning bayesiano de XGBoost, RandomForest e LogReg no alvo de risco, com split e
validação cruzada por rota. Número de trials e timeout ficam em `config.yaml > optuna`.
Resultados e matrizes de confusão vão para `results/risco/`.

In [ ]:
train_risk.otimizado(features_df, cfg, plot=True)